# Imputation for Dynamical Systems: A Crash Course
This tutorial serves as an introduction to the paper {cite}`fim_imputation` and its accompanying code. This page shortly explains the paper and the usage of our trained models. The next pages describe the features seen here in more detail and will allow you to use our models on your own data!
## Introduction 

Dynamical systems are mathematical systems that change with time according to a fixed evolution rule, and serve as representational and analytical tools for  
phenomena which generate patterns that change over time.
Very often, the recorded changes of these empirical patterns are such that they can be viewed as occurring continuously in time, and thus can be represented mathematically by systems whose evolution rule is defined through differential equations.
Dynamical systems governed by ordinary differential equations (ODEs) correspond to an important subset of these models,
and describe the rate of change of a single parametric function 
$\mathbf{x}: \mathbb{R}^+ \rightarrow \mathbb{R}^D$, 
which represents the state of the ($D$-dimensional) system, as time evolves, by means of a vector field 
$\mathbf{f}: \mathbb{R}^+ \times \mathbb{R}^D  \rightarrow \mathbb{R}^D$.

In equations, we write
```{math}
\mathbf{\dot x}(t) = \mathbf{f}(t, \mathbf{x}(t)), \, \,  \text{where} \, \, \, \mathbf{\dot x}(t) = \frac{d \mathbf{x}(t)}{dt}.
```

We consider the general problem of imputing missing values in time series data, recorded from some empirical process $(\mathbf{y}^*: \mathbb{R}^+ \rightarrow \mathbb{R}^D)$ whose dynamics are *assumed* to be governed by some unknown ODE.
In other words, we assume that both available and missing values in the series $\mathbf{y}^*(\tau_1), \dots, \mathbf{y}^*(\tau_l)$ correspond to the values taken by *the solution* $\mathbf{x}(t)$ of some hidden ODE, at the observation times $\tau_1, \dots, \tau_l$, potentially corrupted by some noise signal of which only a few statistics are known.
Therefore, the goal is to infer the ODE solution $\mathbf{x}(t)$ that best *interpolates* the noisy time series $\mathbf{y}^*(\tau_1), \dots, \mathbf{y}^*(\tau_l)$ *and hence imputes its missing values*.

In lieu of training one complex model on a single empirical process, we trained a neural recognition model *offline* to infer a *large and varied set* of ODE solutions $\mathbf{x}(t)$, from a synthetic dataset that is composed of noisy series of observation on those solutions, displaying different missing value patterns.

## Two Formulations of Imputation Problems

The classical formulation of the imputation problem typically involves different missing patterns and 
we will focus on two of them here. The first one is the so-called *point-wise missing pattern*, where individual vectors in the series randomly lack some of their components. The second one is the *temporal missing pattern*, where certain components of the vectors in the series are missing over consecutive observation times. To handle them, we make the following two simple assumptions.
- for the *point-wise missing pattern*, we assume one can always find a certain time scale $\tau_{\small\text{ simple}}$, or some (sequential) subset of observations, for which the best interpolating ODE solution is  "simple". Furthermore, we assume that the set of all such simple parametric functions can be well-represented by a heuristically constructed synthetic distribution. 
- for *temporal missing pattern*, assume the time series *featuring temporal missing patterns*  
involve more complex interpolating functions, meaning that no such $\tau_{\small\text{ simple}}$ is to be found in this case. Although more complex in nature, we assume that these functions are *locally* "simple", and that they often exhibit generic secular and seasonal structures, which encode important information about the missing values and can be well-represented by a second, synthetic distribution over parametric functions.

These two problem setting are solved by two different models, called $\texttt{FIM-}\ell$ and $\texttt{FIM}$, which we will use in the next pages to solve some toy example problems!


We can see this distinction in action in the following code snippets, which will sketch the usage of these two models on some simple dataset. 

:::{warning}
Since this page is only meant to sketch the capabilities of the models, we import the somewhat lengthy functions prepare_data and prepare_data_base. These can found in the repo or slightly adapted the later pages 
- [Point-wise Missing Pattern](imputation_point_wise.ipynb)
- [Temporal Missing Pattern](imputation_temporal.ipynb)
:::

:::{note}
In practice, you probably want to use your GPU for most computations. To do so you need to put both the tensors containing the data and the model on the device, similarly to the MJP case, since our model supports torch syntax,  you can simple use ```mode.to(device)``` with the appropriate device!
:::

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from transformers.utils import logging
logging.disable_progress_bar()
from datasets.utils.logging import disable_progress_bar 
disable_progress_bar()

%matplotlib inline

import torch
from datasets import load_dataset
from imputation_tutorial_helper import visualize_prediction

from fim.models.imputation_pointwise import FIMImpPointBase, FIMImpPoint, StaticWindowing, ImputationConcepts

We will now load some toy data from one dimension of a Roessler attractor. 

In [ ]:
data = load_dataset("FIM4Science/roessler-example", download_mode="force_redownload", name="default")["train"]
data.set_format("torch")

We corrupt the data with noise and randomly drop a percentage of observations, which we will impute. Note that we indicate these points by a mask, where `True` corresponds to dropped points. 

In [ ]:
B = 1
T = 4096
D = 1

noise = 0.05
missing_percentage = 0.1

obs_times = data["t"][:] # [T]

obs_values_clean = data["x"][:] # [T]
multiplicative_error = 1 + torch.normal(0, noise, size=obs_values_clean.shape)
obs_values_noisy = obs_values_clean * multiplicative_error

obs_mask = torch.bernoulli(torch.ones_like(obs_times) * missing_percentage).type(torch.bool)

We impute the missing observations by evaluating the estimated imputing function at the these timestamps.

In [ ]:
evaluation_times = obs_times[obs_mask == 1] # [G]
G = evaluation_times.shape[0]

To apply $\texttt{FIM-}\ell$, we load a pre-trained base model, an instance of the `FIMImpPointBase` class, which has been trained to impute short, one-dimensional time series. 

In [ ]:
fim_imp_pointwise_base = FIMImpPointBase.from_pretrained("FIM4Science/fim-imp-pointwise-base")

Although one-dimensional, this Roessler attractor time series is too long and complicated for the the pre-trained `fim_imp_pointwise_base` to impute. We therefore have to define a scheme to split this time series into shorter trajectories. `fim_imp_pointwise_base` has been trained with up to $128$ observations, so the size of each window should not exceed this threshold by too much.

Such scheme can be implemented in an instance of the `Windowing` base class. Here we use `StaticWindowing`, which chunks the observations into overlapping windows (ignoring the masked values) and implements linear interpolation for the outputs of `fim_imp_pointwise_base` on their overlaps. 

In [ ]:
windowing = StaticWindowing(windows_count=64, overlap_percentage=0.3)

The `FIMImpPoint` class wraps `fim_imp_pointwise_base`, the windowing scheme, and an optional denoising strategy. 
If provided, it denoises the inputs, splits the inputs according to the windowing scheme, applies the pre-trained `fim_imp_pointwise_base` separately for each dimension and window, and combines the outputs, again according to the specified strategy. 

In [ ]:
fim_imp_pointwise = FIMImpPoint(fim_imp_pointwise_base=fim_imp_pointwise_base, windowing=windowing)

We can finally apply the model to impute the missing values. 

In [ ]:
obs_times = obs_times.view(B, T, 1)
obs_values_noisy = obs_values_noisy.view(B, T, D)
obs_mask = obs_mask.view(B, T, 1)
evaluation_times = evaluation_times.view(B, G, 1)

with torch.no_grad():
    output = fim_imp_pointwise.forward(
        obs_times, obs_values_noisy, evaluation_times, obs_mask
    )
    assert isinstance(output, ImputationConcepts)

`ImputationConcepts` is a dataclass containing the relevant outputs. 

For this tutorial, the reconstructed values at the `evaluation_times` are of interest. 

In [ ]:
print(f"Evaluation times shape: {output.evaluation_times.shape}") # [B, G, 1], same as evaluation_times
print(f"Reconstructed values shape: {output.reconstructed_values.shape}") # [B, G, D]

We visualize the data and the predicted values below. 

In [ ]:
visualize_prediction(
    obs_times.flatten(), 
    obs_values_clean.flatten(), 
    obs_values_noisy.flatten(), 
    obs_mask.flatten(),
    output.evaluation_times.flatten(), 
    output.reconstructed_values.flatten(), 
)

See the next chapter to see how we can use the $\texttt{FIM-}\ell$ model to infer the dynamic at play in detail!

Similarly, we can use the $\texttt{FIM}$ model to impute values in the case of a temporal missing pattern i.e. when values are missing over a longer timeframe for which we no longer assume a simple structure of the underlying dynamic. This part can be found [here](imputation_temporal.ipynb).